# 5.1. Multilayer Perceptrons
D2L의 Multilayer Perceptrons장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Layer, Module, Model의 관계

지금까지 우리는 `nn.Linear`같은 layer를 사용해서 신경망을 만들었다. 하지만 실제 신경망은 여러 layer를 묶어서 구성한다.

```text
입력 X
 ↓
Linear
 ↓
ReLU
 ↓
Linear
 ↓
출력
```

PyTorch에서는 이런 구성 요소들을 모두 Module이라는 단위로 다룬다

Module은 하나의 layer일수도 있고, 여러 layer를 묶은 블록일 수도 있고, 전체 모델일 수도 있다.

큰 신경망은 작은 Module들을 조립해서 만든다고 생각하면 된다.

D2L에서 강조하는 건   
layer -> 여러 layer를 묶은 module -> 여러 module을 묶은 더 큰 model이라는 계층 구조이다.

복잡한 ResNet 같은 모델도 결국 이 구조를 반복해서 쌓는다. 

## 2. 가장 간단한 모델구성 nn.Sequential

`nn.Sequential`은 layer들을 순서대로 실행하는 모델을 쉽게 만드는 방법이다.

예를 들어서 아래 MLP를 생각해보면

```text
[batch_size, 20]

 ↓ Linear(20, 256)

[batch_size, 256]

 ↓ ReLU

[batch_size, 256]

 ↓ Linear(256, 10)

[batch_size, 10]
```

nn.Sequential 안에 layer들을 실행할 순서대로 넣으면 된다.

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

X = torch.rand(2, 20)

net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

y_hat = net(X)

print("입력 shape:", X.shape)
print("출력 shape:", y_hat.shape)

입력 shape: torch.Size([2, 20])
출력 shape: torch.Size([2, 10])


Sequential은 내부적으로 아래 작업을 한다.

```text
X
 ↓
Linear(20, 256)
 ↓
ReLU
 ↓
Linear(256, 10)
 ↓
출력
```
이전 layer의 출력이 다음 layer의 입력으로 들어간다.

$$
X_1 = Linear_1(X)
$$

$$
X_2 = ReLU(X_1)
$$

$$
Y = Linear_2(X_2)
$$

`nn.Sequential`은 이 과정을 자동으로 연결해준다. `Sequential` 자체도 `nn.Module`이고, 내부에 다른 `Module`들을 순서대로 보관하고 실행한다고 한다.

## 3. MLP Module 만들기

`nn.Sequential`을 사용하지 않고 우리가 직접 모델을 정의할 수도 있다.

PyTorch에서 직접 모델을 만들 때는 보통 다음 형태를 사용한다고 한다.

```py
class 모델이름(nn.Module):

    def __init__(self): # 모델이 어떤 layer를 가지고 있을지
        super().__init__()

        # 사용할 layer 정의

    def forward(self, X): # 입력 데이터가 그 layer들을 어떤 순서와 방식으로 통과할지 정의

        # 데이터가 어떻게 layer들을 통과할지 정의
```

In [3]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        X = self.hidden(X)
        X = F.relu(X)
        X = self.out(X)

        return X

In [4]:
net = MLP()

y_hat = net(X)

print(y_hat.shape)

torch.Size([2, 10])


실행되는 과정은 이렇다

```text
X [2, 20]

 ↓ self.hidden(X)

[2, 256]

 ↓ ReLU

[2, 256]

 ↓ self.out(X)

[2, 10]
```

전에 썼던 `nn.Sequential`과 결과적으로 같은 MLP이다. 

D2L에서도 사용자 정의 Module에서 기본적으로 `__init__`에서 layer를 정의하고 `forward`에서 계산 흐름을 정의하면, 역전파와 gradient 계산 등은 PyTorch의 autograd가 처리한다고 말한다.